In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Yelp Data Cleaning and Transformation").getOrCreate()

raw_data = spark.read.csv(
    "s3://yelp-final-raja/yelp_database.csv",
    header=True,
    inferSchema=True
)

raw_data.printSchema()
raw_data.show(10)

VBox()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
3,application_1732050930587_0004,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- ID: integer (nullable = true)
 |-- Time_GMT: string (nullable = true)
 |-- Phone: string (nullable = true)
 |-- Organization: string (nullable = true)
 |-- OLF: string (nullable = true)
 |-- Rating: double (nullable = true)
 |-- NumberReview: integer (nullable = true)
 |-- Category: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- CountryCode: string (nullable = true)
 |-- State: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Street: string (nullable = true)
 |-- Building: string (nullable = true)

+---+--------------+-----------+--------------------+----+------+------------+--------+-------+-----------+-----+--------------+--------------+--------+
| ID|      Time_GMT|      Phone|        Organization| OLF|Rating|NumberReview|Category|Country|CountryCode|State|          City|        Street|Building|
+---+--------------+-----------+--------------------+----+------+------------+--------+-------+-----------+-----+--------------+-------------

## Remove duplicates based on 'ID' and 'Organization'

In [2]:

cleaned_data = raw_data.dropDuplicates(["ID", "Organization"])

cleaned_data.show(10)


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---+--------------+-----------+--------------------+----+------+------------+--------+-------+-----------+-----+---------+----------------+--------+
| ID|      Time_GMT|      Phone|        Organization| OLF|Rating|NumberReview|Category|Country|CountryCode|State|     City|          Street|Building|
+---+--------------+-----------+--------------------+----+------+------------+--------+-------+-----------+-----+---------+----------------+--------+
|124|3/12/2021 2:24|12516218501|         Dragon City|NULL|   3.5|          39|Delivery|    USA|         US|   AL|   Daphne|     2101 US Hwy|    2101|
|171|3/12/2021 2:25|12516211112|      Janino's Pizza|NULL|   3.5|          30|Delivery|    USA|         US|   AL|   Daphne| 28567 County Rd|   28567|
|179|3/12/2021 2:25|12516264065|Roll & Go Sushi A...|NULL|   4.0|          10|Delivery|    USA|         US|   AL|   Daphne|            1410|    1410|
|187|3/12/2021 2:25|18503614888|       Marco's Pizza|NULL|   3.0|          25|Delivery|    USA|     

## Drop rows with nulls in critical columns

In [3]:
from pyspark.sql.functions import col

cleaned_data = cleaned_data.dropna(subset=["ID", "Organization", "Rating", "State", "City", "Category"])

cleaned_data = cleaned_data.filter((col("Rating") >= 0) & (col("Rating") <= 5))

cleaned_data.show(10)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---+--------------+-----------+--------------------+----+------+------------+--------+-------+-----------+-----+---------+----------------+--------+
| ID|      Time_GMT|      Phone|        Organization| OLF|Rating|NumberReview|Category|Country|CountryCode|State|     City|          Street|Building|
+---+--------------+-----------+--------------------+----+------+------------+--------+-------+-----------+-----+---------+----------------+--------+
|124|3/12/2021 2:24|12516218501|         Dragon City|NULL|   3.5|          39|Delivery|    USA|         US|   AL|   Daphne|     2101 US Hwy|    2101|
|171|3/12/2021 2:25|12516211112|      Janino's Pizza|NULL|   3.5|          30|Delivery|    USA|         US|   AL|   Daphne| 28567 County Rd|   28567|
|179|3/12/2021 2:25|12516264065|Roll & Go Sushi A...|NULL|   4.0|          10|Delivery|    USA|         US|   AL|   Daphne|            1410|    1410|
|187|3/12/2021 2:25|18503614888|       Marco's Pizza|NULL|   3.0|          25|Delivery|    USA|     

## Select and rename relevant columns for recommendation system

In [ ]:

processed_data = cleaned_data.select(
    col("ID").alias("UserID"),
    col("Organization").alias("ItemID"),
    col("Rating"),
    col("NumberReview"),
    col("State"),
    col("City"),
    col("Category")
)

processed_data.show(10)


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------+--------------------+------+------------+-----+---------+--------+
|UserID|              ItemID|Rating|NumberReview|State|     City|Category|
+------+--------------------+------+------------+-----+---------+--------+
|   124|         Dragon City|   3.5|          39|   AL|   Daphne|Delivery|
|   171|      Janino's Pizza|   3.5|          30|   AL|   Daphne|Delivery|
|   179|Roll & Go Sushi A...|   4.0|          10|   AL|   Daphne|Delivery|
|   187|       Marco's Pizza|   3.0|          25|   FL|Pensacola|Delivery|
|   196|Santino's Pizza &...|   4.0|           5|   FL|   Milton|Delivery|
|   220|                 KFC|   3.0|           4|   AL| Saraland|Delivery|
|   246|   Godfather's Pizza|   4.0|          17|   AL|   Mobile|Delivery|
|   270|      Domino's Pizza|   2.0|          10|   AL|   Daphne|Delivery|
|   285|Hungry Howie's Pizza|   3.5|          11|   AL|   Mobile|Delivery|
|   291|        Shang Hai II|   2.5|          23|   FL|Pensacola|Delivery|
+------+-----------------

## Save processed data to S3 as a single CSV file

In [6]:
processed_data.coalesce(1).write.csv(
    "s3://yelp-final-raja/processed_yelp_data_single/",
    mode="overwrite",
    header=True
)


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…